# SupportSync AI - Multi-Agent Customer Support Desk (LangGraph)
Agent 1 classifies, Agent 2 answers from FAQ KB, Agent 3 escalates with ticket + alert.
Run cells top to bottom.

In [28]:
!pip install -q langgraph langchain-core langchain-google-genai pydantic python-dotenv pandas

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = ""   # https://aistudio.google.com
os.environ["LLM_PROVIDER"] = "gemini-3.5-flash-lite"
os.environ["GEMINI_MODEL"] = "gemini-3.5-flash-lite"
os.environ["DRY_RUN"] = "true"   # emails saved to outbox/ instead of sent
!mkdir -p src data outbox

A subdirectory or file -p already exists.
Error occurred while processing: -p.
A subdirectory or file src already exists.
Error occurred while processing: src.
A subdirectory or file data already exists.
Error occurred while processing: data.
A subdirectory or file outbox already exists.
Error occurred while processing: outbox.


## 1. Knowledge base and test data

In [31]:
%%writefile data/knowledge_base.csv
category,question,answer
billing,How do I get a refund?,"Refunds are processed within 5-7 working days after approval. Requests must be made within 14 days of purchase. Reply with your order number to start the process."
billing,Why was I charged twice?,"Duplicate charges are usually pending authorizations that disappear in 3-5 working days. If the second charge stays after 5 days please share your order number and we will investigate."
billing,Where can I download my invoice?,"Log in to your account, go to Billing > Invoices and click Download PDF next to the invoice you need."
billing,What payment methods do you accept?,"We accept Visa, Mastercard, bank transfer, JazzCash and Easypaisa."
billing,How do I change my billing plan?,"Go to Settings > Subscription > Change Plan. Changes take effect from the next billing cycle."
technical,I forgot my password. How do I reset it?,"Click Forgot Password on the login page and enter your email. A reset link is sent within 2 minutes and is valid for 30 minutes. Check your spam folder too."
technical,The app keeps crashing. What should I do?,"Please update to the latest version, clear the app cache, and restart your device. If the problem continues send us your device model and app version."
technical,I cannot log in to my account.,"Make sure your email is typed correctly and caps lock is off. After 5 failed attempts the account locks for 15 minutes. Use Forgot Password if needed."
technical,How do I enable two-factor authentication?,"Go to Settings > Security > Two-Factor Authentication and follow the steps to link an authenticator app."
technical,The website is loading slowly.,"Try clearing your browser cache or using another browser. Check status.yourcompany.com for any ongoing incidents."
general,What are your support hours?,"Our support team is available Monday to Saturday 9:00 AM - 9:00 PM PKT. The AI assistant replies 24/7."
general,How can I contact a human agent?,"Reply to this email asking for a human agent and your request will be assigned to our support team within 24 hours."
general,Where is my order?,"You can track your order under My Orders > Track. Standard delivery takes 3-5 working days within Pakistan."
general,Do you offer discounts for students?,"Yes, students get 20% off with a valid student ID. Email your ID to the support address to receive a code."
general,How do I delete my account?,"Go to Settings > Account > Delete Account. Data is permanently removed after 30 days."
complaint,I am unhappy with your service.,"We are sorry about your experience. Complaints are reviewed by a senior support agent within 24 hours."

Overwriting data/knowledge_base.csv


In [17]:
%%writefile data/test_queries.csv
id,name,email,message,expected_category,expected_path
1,Ali Khan,ali@example.com,"I forgot my password, how can I reset it?",technical,Auto reply
2,Sara Ahmed,sara@example.com,"Please send me my invoice for last month.",billing,Auto reply
3,Usman Raza,usman@example.com,"What are your support timings?",general,Auto reply
4,Hina Malik,hina@example.com,"I was charged twice and nobody is replying. This is unacceptable, I want my money back or I will go to court!",complaint,Escalate (high)
5,Bilal Sheikh,bilal@example.com,"The app crashes every time I open it since yesterday's update.",technical,Auto reply
6,Ayesha Noor,ayesha@example.com,"Can I change the delivery address of my custom order placed in Dubai branch?",general,Escalate (not in KB)
7,Zain Ali,zain@example.com,"Your support staff was rude to me yesterday.",complaint,Escalate

Overwriting data/test_queries.csv


## 2. Source code (state, LLM, retriever, tools, agents, graph)

In [32]:
%%writefile src/__init__.py
# package

Overwriting src/__init__.py


In [33]:
%%writefile src/state.py
import operator
from typing import Annotated, TypedDict


class SupportState(TypedDict, total=False):
    # input
    ticket_id: str
    name: str
    email: str
    subject: str
    message: str
    # Agent 1 output
    category: str      # billing | technical | complaint | general
    priority: str      # low | medium | high
    sentiment: str     # positive | neutral | negative
    summary: str
    # Agent 2 output
    kb_context: str
    reply: str
    resolved: bool
    confidence: int
    # Agent 3 output
    ticket_summary: str
    suggested_action: str
    acknowledgement: str
    # result
    status: str        # "Resolved by AI" | "Escalated"
    trace: Annotated[list, operator.add]

Overwriting src/state.py


In [ ]:
%%writefile src/llm.py
import os
from dotenv import load_dotenv

load_dotenv()


def get_llm(temperature: float = 0.2):
    provider = os.getenv("LLM_PROVIDER", "gemini").lower()
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), temperature=temperature)
    from langchain_google_genai import ChatGoogleGenerativeAI
    return ChatGoogleGenerativeAI(model=os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite"), temperature=temperature)

Overwriting src/llm.py


In [35]:
%%writefile src/kb.py
"""Tiny keyword retriever over the FAQ knowledge base (CSV). Good enough for a small KB;
swap for a vector store (Chroma/FAISS) if the KB grows."""
import re
from pathlib import Path
import pandas as pd

KB_PATH = Path(__file__).resolve().parent.parent / "data" / "knowledge_base.csv"
STOP = set("a an the is are was i my me to of for and or in on it do how can you your we our with this that what when why".split())


def _tokens(text: str) -> set:
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP}


def retrieve(query: str, category: str | None = None, k: int = 4) -> str:
    df = pd.read_csv(KB_PATH)
    q = _tokens(query)
    scored = []
    for _, r in df.iterrows():
        score = len(q & _tokens(f"{r['question']} {r['answer']}"))
        if category and r["category"] == category:
            score += 0.5
        scored.append((score, r))
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [r for s, r in scored[:k] if s >= 1]
    if not top:
        return "NO RELEVANT KNOWLEDGE BASE ENTRIES FOUND."
    return "\n\n".join(f"[{r['category']}] Q: {r['question']}\nA: {r['answer']}" for r in top)

Overwriting src/kb.py


In [36]:
%%writefile src/tools.py
"""Side-effect tools: send email, create ticket, log. DRY_RUN=true keeps everything local."""
import csv, json, os, smtplib
from datetime import datetime
from email.message import EmailMessage
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
ROOT = Path(__file__).resolve().parent.parent
DATA = ROOT / "data"
OUTBOX = ROOT / "outbox"
TICKET_FIELDS = ["ticket_id", "created_at", "customer", "email", "category", "priority", "sentiment",
                 "summary", "suggested_action", "status", "assigned_to"]
LOG_FIELDS = ["ticket_id", "received_at", "customer", "email", "category", "priority", "status", "message"]


def send_email(to: str, subject: str, body: str) -> str:
    if os.getenv("DRY_RUN", "true").lower() == "true":
        OUTBOX.mkdir(exist_ok=True)
        with open(OUTBOX / "outbox.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps({"time": datetime.now().isoformat(), "to": to, "subject": subject, "body": body}, ensure_ascii=False) + "\n")
        return f"[DRY_RUN] email saved for {to}"
    msg = EmailMessage()
    msg["From"], msg["To"], msg["Subject"] = os.environ["SMTP_USER"], to, subject
    msg.set_content(body)
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as s:
        s.login(os.environ["SMTP_USER"], os.environ["SMTP_APP_PASSWORD"])
        s.send_message(msg)
    return f"email sent to {to}"


def _append(path: Path, fields: list, row: dict):
    new = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow(row)


def create_ticket(row: dict) -> str:
    _append(DATA / "tickets.csv", TICKET_FIELDS, row)
    return f"ticket {row['ticket_id']} created"


def log_resolved(row: dict) -> str:
    _append(DATA / "log.csv", LOG_FIELDS, row)
    return f"logged {row['ticket_id']}"

Overwriting src/tools.py


In [37]:
%%writefile src/agents.py
from typing import Literal
from pydantic import BaseModel, Field
from .llm import get_llm
from .kb import retrieve
from .state import SupportState


# ---------- structured outputs ----------
class Classification(BaseModel):
    category: Literal["billing", "technical", "complaint", "general"]
    priority: Literal["low", "medium", "high"]
    sentiment: Literal["positive", "neutral", "negative"]
    summary: str = Field(description="One short sentence summarising the query")


class Reply(BaseModel):
    reply: str = Field(description="Email body to the customer, max 120 words")
    resolved: bool = Field(description="True only if the knowledge base clearly answers the question")
    confidence: int = Field(description="0-100")


class Escalation(BaseModel):
    ticket_summary: str
    suggested_action: str
    acknowledgement: str = Field(description="Short email (max 60 words) telling the customer a human will contact them within 24 hours")


# ---------- Agent 1: Classifier ----------
CLASSIFIER_PROMPT = """You are Agent 1, the Query Classifier of a customer support desk.
Classify the customer's message.
- billing: refunds, invoices, payments, charges
- technical: errors, bugs, login, app/website problems
- complaint: angry/unhappy customers, rude staff, repeated failures, legal threats
- general: everything else
priority=high if the customer is angry, mentions legal action, fraud, money lost, or the service is completely down."""


def classifier_agent(state: SupportState) -> dict:
    llm = get_llm(0).with_structured_output(Classification)
    r = llm.invoke([("system", CLASSIFIER_PROMPT),
                    ("human", f"Subject: {state.get('subject','')}\nMessage: {state['message']}")])
    return {"category": r.category, "priority": r.priority, "sentiment": r.sentiment, "summary": r.summary,
            "trace": [f"Agent 1 (Classifier): category={r.category}, priority={r.priority}, sentiment={r.sentiment}"]}


# ---------- Agent 2: Reply generator (RAG over KB) ----------
REPLY_PROMPT = """You are Agent 2, the Reply Writer of a customer support desk.
Answer ONLY using the KNOWLEDGE BASE provided. Never invent policies, prices, dates or promises.
Be polite, address the customer by name, keep it under 120 words, sign off as "SupportSync Support Team".
If the knowledge base does not clearly answer the question set resolved=false and confidence below 60."""


def reply_agent(state: SupportState) -> dict:
    kb = retrieve(state["message"], state.get("category"))
    llm = get_llm(0.3).with_structured_output(Reply)
    r = llm.invoke([("system", REPLY_PROMPT),
                    ("human", f"Customer name: {state['name']}\nCategory: {state['category']}\n"
                              f"Message: {state['message']}\n\nKNOWLEDGE BASE:\n{kb}")])
    return {"kb_context": kb, "reply": r.reply, "resolved": r.resolved, "confidence": r.confidence,
            "trace": [f"Agent 2 (Reply): resolved={r.resolved}, confidence={r.confidence}"]}


# ---------- Agent 3: Escalation ----------
ESCALATION_PROMPT = """You are Agent 3, the Escalation Agent of a customer support desk.
Prepare a hand-off for a human support agent: a 2-3 sentence ticket summary (what happened, what the customer wants),
one concrete suggested next action, and a short polite acknowledgement email for the customer signed "SupportSync Support Team"."""


def escalation_agent(state: SupportState) -> dict:
    llm = get_llm(0.2).with_structured_output(Escalation)
    r = llm.invoke([("system", ESCALATION_PROMPT),
                    ("human", f"Customer: {state['name']} ({state['email']})\nCategory: {state['category']}\n"
                              f"Priority: {state['priority']}\nSentiment: {state['sentiment']}\nMessage: {state['message']}")])
    return {"ticket_summary": r.ticket_summary, "suggested_action": r.suggested_action,
            "acknowledgement": r.acknowledgement,
            "trace": ["Agent 3 (Escalation): ticket summary prepared"]}

Overwriting src/agents.py


In [38]:
%%writefile src/graph.py
import os
from datetime import datetime
from langgraph.graph import StateGraph, START, END
from .state import SupportState
from .agents import classifier_agent, reply_agent, escalation_agent
from . import tools

CONFIDENCE_THRESHOLD = 60


# ---------- action nodes ----------
def send_reply_node(state: SupportState) -> dict:
    res = tools.send_email(state["email"], f"Re: {state.get('subject', 'Your request')}", state["reply"])
    tools.log_resolved({"ticket_id": state["ticket_id"], "received_at": datetime.now().isoformat(),
                        "customer": state["name"], "email": state["email"], "category": state["category"],
                        "priority": state["priority"], "status": "Resolved by AI", "message": state["message"]})
    return {"status": "Resolved by AI", "trace": [f"Send reply: {res}"]}


def ticket_node(state: SupportState) -> dict:
    t = tools.create_ticket({
        "ticket_id": state["ticket_id"], "created_at": datetime.now().isoformat(), "customer": state["name"],
        "email": state["email"], "category": state["category"], "priority": state["priority"],
        "sentiment": state["sentiment"], "summary": state["ticket_summary"],
        "suggested_action": state["suggested_action"], "status": "Open", "assigned_to": ""})
    team = os.getenv("SUPPORT_TEAM_EMAIL", "support-team@yourcompany.com")
    a = tools.send_email(
        team, f"[{state['priority'].upper()}] Escalated ticket {state['ticket_id']}",
        f"Category: {state['category']}\nCustomer: {state['name']} ({state['email']})\n\n"
        f"Summary: {state['ticket_summary']}\n\nSuggested action: {state['suggested_action']}\n\n"
        f"Original message:\n{state['message']}")
    c = tools.send_email(state["email"], f"We received your request ({state['ticket_id']})", state["acknowledgement"])
    return {"status": "Escalated", "trace": [f"Ticket: {t}", f"Team alert: {a}", f"Customer ack: {c}"]}


# ---------- routers ----------
def route_after_classify(state: SupportState) -> str:
    # Complaints / high priority skip auto-reply and go straight to a human
    if state["category"] == "complaint" or state["priority"] == "high":
        return "escalate"
    return "reply"


def route_after_reply(state: SupportState) -> str:
    if (not state["resolved"]) or state["confidence"] < CONFIDENCE_THRESHOLD:
        return "escalate"
    return "send"


def build_graph():
    g = StateGraph(SupportState)
    g.add_node("classify", classifier_agent)
    g.add_node("reply", reply_agent)
    g.add_node("send_reply", send_reply_node)
    g.add_node("escalation_agent", escalation_agent)
    g.add_node("create_ticket", ticket_node)

    g.add_edge(START, "classify")
    g.add_conditional_edges("classify", route_after_classify, {"reply": "reply", "escalate": "escalation_agent"})
    g.add_conditional_edges("reply", route_after_reply, {"send": "send_reply", "escalate": "escalation_agent"})
    g.add_edge("escalation_agent", "create_ticket")
    g.add_edge("send_reply", END)
    g.add_edge("create_ticket", END)
    return g.compile()


def run_query(name: str, email: str, message: str, subject: str = "Support Request", graph=None) -> dict:
    graph = graph or build_graph()
    init = {"ticket_id": "TKT-" + datetime.now().strftime("%m%d%H%M%S%f")[:12], "name": name, "email": email,
            "subject": subject, "message": message, "trace": []}
    return graph.invoke(init)

Overwriting src/graph.py


## 3. Draw the graph

In [39]:
from src.graph import build_graph
graph = build_graph()
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	reply(reply)
	send_reply(send_reply)
	escalation_agent(escalation_agent)
	create_ticket(create_ticket)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	classify -. &nbsp;escalate&nbsp; .-> escalation_agent;
	classify -.-> reply;
	escalation_agent --> create_ticket;
	reply -. &nbsp;escalate&nbsp; .-> escalation_agent;
	reply -. &nbsp;send&nbsp; .-> send_reply;
	create_ticket --> __end__;
	send_reply --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 4. Try one query

In [41]:
from src.graph import run_query
res = run_query("Ali Khan", "ali@example.com", "I forgot my password, how can I reset it?", graph=graph)
print(res["category"], res["priority"], res["status"])
for t in res["trace"]: print(" ->", t)
print(res.get("reply"))

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


technical low Resolved by AI
 -> Agent 1 (Classifier): category=technical, priority=low, sentiment=neutral
 -> Agent 2 (Reply): resolved=True, confidence=100
 -> Send reply: [DRY_RUN] email saved for ali@example.com
Hello Ali Khan, to reset your password, click Forgot Password on the login page and enter your email. A reset link will be sent within 2 minutes and is valid for 30 minutes. Please check your spam folder as well if you do not see it. SupportSync Support Team


## 5. Run all 7 test queries

In [42]:
import pandas as pd
for _, r in pd.read_csv("data/test_queries.csv").iterrows():
    o = run_query(r["name"], r["email"], r["message"], graph=graph)
    print(f"{r['id']} | {o['category']:9} | {o['priority']:6} | {o['status']:15} | expected: {r['expected_path']}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


1 | technical | low    | Resolved by AI  | expected: Auto reply


c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


2 | billing   | low    | Resolved by AI  | expected: Auto reply


c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


...

5 | technical | medium | Resolved by AI  | expected: Auto reply


c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


6 | general   | low    | Escalated       | expected: Escalate (not in KB)


c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
c:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


7 | complaint | high   | Escalated       | expected: Escalate


## 6. Results: tickets, log and emails

In [43]:
print(pd.read_csv("data/tickets.csv").to_string())
print(pd.read_csv("data/log.csv").to_string())
print(pd.read_json("outbox/outbox.jsonl", lines=True).to_string())

          ticket_id                  created_at     customer               email   category priority sentiment                                                                                                                                                                                                                                         summary                                                                                                                                                                suggested_action status  assigned_to
0  TKT-100302480635  2026-10-03T02:48:13.191816   Hina Malik    hina@example.com    billing     high  negative                                           Hina Malik experienced a double charge on her account and is extremely upset due to lack of prior responses. She is demanding a full refund and threatening legal action if not resolved immediately.  Verify the duplicate charge in the billing system, process an immediate refund for the extra transact